# Week 62, Day 4: Exact inference by message passing

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Sum-product on a chain

Complete chain_marginals(phi, psi), where phi is a T by K array of evidence and psi a K by K coupling matrix: compute normalised forward and backward messages and return the T by K array of marginals. Check it against enumeration of all 3 to the power 7 states of a short chain.

In [ ]:
import numpy as np
import itertools

def chain_marginals(phi, psi):
    T, K = phi.shape
    alpha = np.zeros((T, K)); beta = np.ones((T, K))
    alpha[0] = phi[0] / phi[0].sum()
    for t in range(1, T):
        a = phi[t] * (alpha[t - 1] @ psi)
        alpha[t] = a / a.sum()
    for t in range(T - 2, -1, -1):
        b = ...
        beta[t] = b / b.sum()
    m = alpha * beta
    return m / m.sum(1, keepdims=True)

gc = np.random.default_rng(624)
T7, K3 = 7, 3
phi7 = gc.uniform(0.1, 1, (T7, K3))
psi3 = np.array([[0.8, 0.15, 0.05], [0.1, 0.8, 0.1], [0.05, 0.15, 0.8]])
marg_bp = chain_marginals(phi7, psi3)
marg_brute = np.zeros((T7, K3))
for xs in itertools.product(range(K3), repeat=T7):
    w = np.prod([phi7[t, xs[t]] for t in range(T7)]) * np.prod([psi3[xs[t], xs[t + 1]] for t in range(T7 - 1)])
    for t in range(T7):
        marg_brute[t, xs[t]] += w
marg_brute /= marg_brute.sum(1, keepdims=True)
print("largest difference from enumeration:", np.abs(marg_bp - marg_brute).max())
print(f"enumeration summed {K3 ** T7} configurations; message passing did {2 * (T7 - 1)} K-by-K products")

In [ ]:
import numpy as np
def test_brute():
    "chain_marginals matches brute-force enumeration on a short chain with 3 states"
    assert np.allclose(marg_bp, marg_brute, atol=1e-10)
def test_norm():
    "each marginal sums to one"
    assert np.allclose(marg_bp.sum(1), 1)

test_brute()
test_norm()
print("All checks passed")

### Exercise: Tracking engagement

Simulate 600 trials in which engagement switches with probability 0.03 per trial; engaged trials are correct with probability 0.9, disengaged with 0.55. Classify engagement on each trial from its outcome alone (correct means engaged) and from the chain's marginals, and compare accuracies.

In [ ]:
ge = np.random.default_rng(6241)
Tn = 600
z = np.zeros(Tn, int); z[0] = 1
for t in range(1, Tn):
    z[t] = 1 - z[t - 1] if ge.random() < 0.03 else z[t - 1]
p_correct = np.where(z == 1, 0.9, 0.55)
correct = (ge.random(Tn) < p_correct).astype(int)

psi_e = np.array([[0.97, 0.03], [0.03, 0.97]])
phi_e = ...                                            # T x 2: likelihood of each outcome under disengaged (0) and engaged (1)
marg_e = chain_marginals(phi_e, psi_e)
acc_chain = np.mean((marg_e[:, 1] > 0.5) == z)
acc_single = np.mean(correct == z)
print(f"fraction of trials engaged {z.mean():.2f}; accuracy from single outcomes {acc_single:.3f}, from the chain {acc_chain:.3f}")

In [ ]:
import numpy as np
def test_better():
    "the chain's marginals classify engagement better than each trial's outcome alone, by more than 0.1"
    assert acc_chain > acc_single + 0.1 and acc_chain > 0.85

test_better()
print("All checks passed")

### Exercise: Belief propagation on a tree

Complete message(i, j), which computes the message from node i to node j recursively from i's evidence, the edge potential, and the messages into i from its other neighbours. Compute every node's marginal and compare with enumeration.

In [ ]:
edges = [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6)]           # a binary tree of 7 nodes
nbrs = {i: [] for i in range(7)}
for a, b in edges:
    nbrs[a].append(b); nbrs[b].append(a)
gtr = np.random.default_rng(6242)
phi_t = gtr.uniform(0.2, 1, (7, 2))
psi_t = {}
for a, b in edges:
    M = gtr.uniform(0.2, 1, (2, 2)); psi_t[(a, b)] = M; psi_t[(b, a)] = M.T

def message(i, j):
    incoming = np.prod([message(k, i) for k in nbrs[i] if k != j], axis=0) if len(nbrs[i]) > 1 else np.ones(2)
    m = ...
    return m / m.sum()

tree_m = np.array([phi_t[i] * np.prod([message(k, i) for k in nbrs[i]], axis=0) for i in range(7)])
tree_m /= tree_m.sum(1, keepdims=True)

brute_m = np.zeros((7, 2))
for xs in itertools.product((0, 1), repeat=7):
    w = np.prod([phi_t[i, xs[i]] for i in range(7)]) * np.prod([psi_t[(a, b)][xs[a], xs[b]] for a, b in edges])
    for i in range(7):
        brute_m[i, xs[i]] += w
brute_m /= brute_m.sum(1, keepdims=True)
print("largest difference from enumeration:", np.abs(tree_m - brute_m).max())

In [ ]:
import numpy as np
def test_tree():
    "tree_marginals matches enumeration on a 7-node tree"
    assert np.allclose(tree_m, brute_m, atol=1e-10)

test_tree()
print("All checks passed")